# Практика 2. Три источника — одна таблица

**Ситуация.** На практике 1 мы заметили, что на ADSL расторгают договор заметно чаще. Но ADSL сосредоточен в Кургане
и Нижнем Тагиле, поэтому руководитель филиала спрашивает: **дело в технологии или в городе?**
Чтобы ответить, нужна единая таблица «один абонент — одна строка», собранная из трёх систем:

| Файл | Система | Что в нём |
|---|---|---|
| `data/subscribers.csv` | CRM | справочник абонентов |
| `data/tickets.csv` | система обращений | обращения в поддержку за январь–сентябрь |
| `data/billing.xlsx` | биллинг | начисления по месяцам |

Описание столбцов — в `data/README.md`. **Прочитайте его до начала работы.**

**Предупреждение.** Это настоящие выгрузки: системы ведут разные отделы, и данные «грязные».
Ни один файл не читается правильно с первой попытки и с параметрами по умолчанию.
Каждую проблему, которую вы найдёте и исправите, записывайте в **журнал очистки** в конце ноутбука.

**Сдать до 23:59 в день занятия:**
- тест «Практика 2» — 8 числовых ответов, одна попытка, 8 баллов;
- этот ноутбук — 4 балла (критерии в конце).

> ИИ-ассистентами пользоваться можно. Но ассистент не видит ваши файлы целиком, а значит, не знает,
> что в них не так. Код, написанный «вслепую», здесь почти наверняка даст неверный ответ.
> **Сначала смотрите на данные, потом пишите код.**

In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 50)

## Часть 1. Справочник абонентов

1. Загрузите `subscribers.csv`. Посмотрите на **все уникальные значения** текстовых столбцов (`city`, `status`, `tech`), на диапазоны чисел, на форматы `client_id` и `connected`.
2. Приведите к одному виду идентификаторы, города, статусы, технологии.
3. Разберитесь с дублями. В выгрузке могут быть и копии, и старые версии записей: оставьте по каждому абоненту **актуальную**.
4. Разберитесь с `nps` и `traffic_gb`: всё ли там похоже на то, что обещает описание?

*Подсказки:* `df[col].value_counts(dropna=False)`, `df.describe()`, `df.groupby('city')[col].describe()`, `str.strip()`, `str.lower()`, `sort_values` + `drop_duplicates(keep=...)`.

In [19]:
sub = pd.read_csv('data/subscribers.csv')
sub.head()

,client_id,account,city,tariff,speed_mbps,tech,tv_package,connected,traffic_gb,nps,status,updated_at
0,RT-203160,41203160,Тюмень,Семейный 300,300,xPON,нет,2019-05-24,432.9,9.0,активен,2026-10-14 03:34:18
1,RT-201441,69201441,Пермь,Базовый 100,100,xpon,да,13.12.2019,496844.8,8.0,активен,2026-10-03 04:32:58
2,RT-202591,82202591,Екатеринбург,Базовый 100,100,xPON,нет,45434,229.8,8.0,расторгнут,2026-10-14 16:15:52
3,RT-201121,53201121,Курган,Семейный 300,300,FTTB,нет,2021-04-01,164.4,8.0,расторгнут,2026-10-09 08:16:17
4,RT-201667,55201667,Екатеринбург,Базовый 100,100,xPON,нет,2021-06-14,328.9,NaN,активен,2026-10-18 23:46:34


In [20]:
df = sub.copy()

In [21]:
for col in ['city', 'status', 'tech']:
    df[col] = df[col].astype(str).str.strip().str.lower()

In [22]:
df = df.sort_values(by=['client_id', 'updated_at'])
df = df.drop_duplicates(subset=['client_id'], keep='last')

In [23]:
df.head()

,client_id,account,city,tariff,speed_mbps,tech,tv_package,connected,traffic_gb,nps,status,updated_at
2817,RT-200006,73200006,тюмень,Игровой 500,500,fttb,нет,2024-03-13,81.4,8.0,активен,2026-10-06 17:43:48
476,RT-200063,39200063,нижний тагил,Базовый 100,100,fttb,да,2024-01-28,328.9,9.0,расторгнут,2026-10-11 12:31:46
324,RT-200070,98200070,екатеринбург,Базовый 100,100,fttb,да,2024-05-28,167.1,8.0,активен,2026-10-18 18:54:21
580,RT-200084,22200084,курган,Гигабит 1000,1000,xpon,да,2022-01-08,1723.3,9.0,активен,2026-10-04 14:20:12
1123,RT-200110,28200110,пермь,Базовый 100,100,xpon,нет,2021-09-13,418816.0,10.0,активен,2026-10-06 16:16:44


In [24]:
print("--- Уникальные значения NPS ---")
display(df['nps'].value_counts(dropna=False))

print("\n--- Описательная статистика traffic_gb ---")
display(df['traffic_gb'].describe())

print("\n--- Статистика traffic_gb в разрезе городов ---")
display(df.groupby('city')['traffic_gb'].describe())

--- Уникальные значения NPS ---


nps
 8.0     643
 7.0     626
 9.0     566
 6.0     400
 10.0    376
 5.0     199
 NaN     127
-1.0     116
 4.0      83
 99.0     61
 3.0      22
 2.0      15
 1.0       8
 0.0       5
Name: count, dtype: int64


--- Описательная статистика traffic_gb ---


count    3.247000e+03
mean     7.919209e+04
std      2.131742e+05
min      9.900000e+00
25%      2.164000e+02
50%      3.881000e+02
75%      9.959500e+02
max      2.288333e+06
Name: traffic_gb, dtype: float64


--- Статистика traffic_gb в разрезе городов ---


,count,mean,std,min,25%,50%,75%,max
city,,,,,,,,
chelyabinsk,39.0,431.412821,374.695610,40.8,173.700,275.20,513.150,1625.9
ekaterinburg,46.0,346.858696,298.916512,16.7,131.575,255.20,416.025,1294.6
г. екатеринбург,50.0,383.718000,325.958787,61.0,199.825,320.85,462.375,2077.9
г. курган,15.0,322.386667,161.870102,65.1,242.300,280.00,420.300,700.7
г. пермь,52.0,425312.492308,327078.505772,49049.6,165913.600,330035.20,586470.400,1399296.0
г. тюмень,31.0,489.061290,418.015069,82.7,200.250,333.80,649.500,1994.0
г. челябинск,36.0,344.713889,222.300742,40.3,257.275,301.65,361.000,1290.8
екатеринбург,1008.0,409.614782,337.075282,9.9,189.850,304.05,517.375,3509.9
курган,256.0,398.391016,305.916652,22.0,177.375,316.65,528.300,1986.2


## Часть 2. Обращения в поддержку

1. Загрузите `tickets.csv`. Посмотрите описание: кодировка, разделитель.
2. Если `read_csv` ругается на часть строк — **не пропускайте их молча**. Посмотрите, что в них не так, и подумайте, можно ли их восстановить.
3. Приведите идентификатор абонента к формату справочника.
4. Уберите то, что не является обращением абонента справочника, и двойную регистрацию одного обращения.
5. Посчитайте число обращений на абонента и присоедините к справочнику так, чтобы **абоненты без обращений не потерялись**.

*Подсказки:* `encoding=`, `sep=`, `on_bad_lines=`, модуль `csv`, `groupby(...).created_at.diff()`, `merge(how=...)`, `fillna`.

In [27]:
df_tickets = pd.read_csv('data/tickets.csv', sep=';', encoding='cp1251', on_bad_lines='warn')

print("Размер датафрейма:", df_tickets.shape)
display(df_tickets.head())

Размер датафрейма: (2241, 6)


/var/folders/n3/m9rvhfd50gl19w81qyftmtzm0000gn/T/ipykernel_59363/1037270549.py:1: ParserWarning: Skipping line 126: expected 6 fields, saw 7
Skipping line 183: expected 6 fields, saw 7
Skipping line 243: expected 6 fields, saw 7
Skipping line 383: expected 6 fields, saw 7
Skipping line 457: expected 6 fields, saw 7
Skipping line 616: expected 6 fields, saw 7
Skipping line 701: expected 6 fields, saw 7
Skipping line 1133: expected 6 fields, saw 7
Skipping line 1312: expected 6 fields, saw 7
Skipping line 1379: expected 6 fields, saw 7
Skipping line 1622: expected 6 fields, saw 7
Skipping line 2109: expected 6 fields, saw 7
Skipping line 2184: expected 6 fields, saw 7
Skipping line 2231: expected 6 fields, saw 7

  df_tickets = pd.read_csv('data/tickets.csv', sep=';', encoding='cp1251', on_bad_lines='warn')


,ticket_id,client,created_at,category,channel,comment
0,500001,201275,2026-01-01 01:49:18,Нет интернета,чат,пропадает связь вечером
1,500002,200416,2026-01-01 03:51:31,Оплата,офис,вопрос по счёту
2,500003,203053,2026-01-01 08:41:49,Подключение ТВ,телефон,хочет подключить ТВ
3,500004,TEST-0007,2026-01-01 09:02:46,Прочее,телефон,консультация
4,500005,200064,2026-01-01 09:03:41,Прочее,приложение,перенос оборудования


In [32]:
with open('data/tickets.csv', 'r', encoding='cp1251') as f:
    lines = f.readlines()
    
print(lines[125].strip())
print(lines[182].strip())
print(lines[183].strip())

print(lines[124].strip())

500125;200399;2026-01-15 14:44:36;Нет интернета;телефон;роутер мигает красным; клиент просит перезвонить
500182;201242;2026-01-21 04:55:17;Подключение ТВ;приложение;не работает приставка; клиент просит перезвонить
500183;RT-201554;2026-01-21 08:37:03;Нет интернета;чат;пропадает связь вечером
500124;RT-200644;2026-01-15 11:43:45;Прочее;телефон;консультация


In [33]:
import csv

rows = []
fixed_count = 0

with open('data/tickets.csv', 'r', encoding='cp1251') as f:
    reader = csv.reader(f, delimiter=';')
    header = next(reader)
    
    for row in reader:
        if len(row) == 6:
            rows.append(row)
        elif len(row) > 6:
            # Склеиваем разорванный комментарий обратно в одно поле
            fixed_row = row[:5] + [';'.join(row[5:])]
            rows.append(fixed_row)
            fixed_count += 1

df_tickets = pd.DataFrame(rows, columns=header)

print(f"Успешно восстановлено строк: {fixed_count}")
print(f"Размер датафрейма: {df_tickets.shape}")
display(df_tickets.head())

Успешно восстановлено строк: 14
Размер датафрейма: (2255, 6)


,ticket_id,client,created_at,category,channel,comment
0,500001,201275,2026-01-01 01:49:18,Нет интернета,чат,пропадает связь вечером
1,500002,200416,2026-01-01 03:51:31,Оплата,офис,вопрос по счёту
2,500003,203053,2026-01-01 08:41:49,Подключение ТВ,телефон,хочет подключить ТВ
3,500004,TEST-0007,2026-01-01 09:02:46,Прочее,телефон,консультация
4,500005,200064,2026-01-01 09:03:41,Прочее,приложение,перенос оборудования


In [34]:
df_tickets = df_tickets.rename(columns={'client': 'client_id'})

print("Приведенные client_id в tickets:")
display(df_tickets['client_id'].head())

Приведенные client_id в tickets:


0       201275
1       200416
2       203053
3    TEST-0007
4       200064
Name: client_id, dtype: str

In [35]:
# 1. Убираем тестовые записи (всё, что содержит 'TEST')
df_tickets = df_tickets[~df_tickets['client_id'].astype(str).str.contains('TEST', case=False, na=False)]

# 2. Приводим дату создания к формату datetime для корректной сортировки
df_tickets['created_at'] = pd.to_datetime(df_tickets['created_at'], errors='coerce')

# 3. Сортируем по клиенту и времени создания обращения
df_tickets = df_tickets.sort_values(by=['client_id', 'created_at'])

# 4. Считаем разницу во времени с предыдущим обращением того же клиента
df_tickets['time_diff'] = df_tickets.groupby('client_id')['created_at'].diff()

# 5. Убираем дубли (двойную регистрацию): оставляем обращение, только если прошло больше 1 минуты с предыдущего (или это первое обращение — NaN)
df_tickets = df_tickets[(df_tickets['time_diff'] > pd.Timedelta(minutes=1)) | (df_tickets['time_diff'].isna())]

# Удаляем временную вспомогательную колонку
df_tickets = df_tickets.drop(columns=['time_diff'])

print(f"Размер датафрейма после очистки от тестов и дублей: {df_tickets.shape}")

Размер датафрейма после очистки от тестов и дублей: (2120, 6)


In [36]:
# 1. Считаем количество обращений по каждому client_id
tickets_count = df_tickets.groupby('client_id').size().reset_index(name='tickets_count')

# 2. Присоединяем к основному справочнику (предполагается, что он называется df) через left join,
# чтобы абоненты без обращений не потерялись
df = df.merge(tickets_count, on='client_id', how='left')

# 3. Заменяем пропуски (NaN) на 0 для тех, кто не обращался в поддержку, и переводим в целые числа
df['tickets_count'] = df['tickets_count'].fillna(0).astype(int)

# Проверяем результат
print("Размер итогового датафрейма абонентов:", df.shape)
display(df[['client_id', 'tickets_count']].head(10))

Размер итогового датафрейма абонентов: (3247, 13)


,client_id,tickets_count
0,RT-200006,0
1,RT-200063,0
2,RT-200070,0
3,RT-200084,0
4,RT-200110,0
5,RT-200119,0
6,RT-200158,0
7,RT-200186,0
8,RT-200204,0
9,RT-200219,0


## Часть 3. Биллинг

1. Откройте `billing.xlsx` и посмотрите на лист **глазами**: где начинается таблица, где заканчивается, что написано вокруг неё.
2. Прочитайте таблицу так, чтобы суммы стали числами.
3. Свяжите начисления со справочником. Подумайте, по какому ключу.
4. Проверьте, что суммы во всех городах в одних единицах.

*Подсказки:* `pd.read_excel(..., header=..., dtype=str)`, `str.replace`, `melt`.

In [37]:
df_billing_raw = pd.read_excel('data/billing.xlsx', header=None)
display(df_billing_raw.head(15))

,0,1,2,3,4,5,6,7,8,9,10
0,Выгрузка из АСР «Биллинг-М». Начисления абонен...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Сформировано: 01.10.2026 03:15. Филиал: Уральс...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Лицевой счёт,Тип клиента,янв.26,фев.26,мар.26,апр.26,май.26,июн.26,июл.26,авг.26,сен.26
4,50201125,ФЛ,"750,00","750,00","750,00","483,00","750,00","750,00","750,00","750,00","750,00"
5,69202258,ФЛ,"55 000,00","55 000,00","55 000,00","55 000,00","55 000,00","55 000,00","55 000,00","55 000,00","55 000,00"
6,33201941,ФЛ,"800,00","800,00","800,00","800,00","800,00","800,00","800,00","800,00","800,00"
7,45202567,ФЛ,"950,00","950,00","950,00","950,00","950,00","950,00","950,00","950,00","950,00"
8,97201898,ФЛ,"44 000,00","44 000,00","44 000,00","44 000,00","44 000,00","44 000,00","44 000,00","44 000,00","44 000,00"
9,58202836,ФЛ,"1 411,00","1 411,00","1 411,00","1 411,00","1 411,00","1 411,00","1 411,00","1 411,00","1 411,00"


In [40]:

df_billing = pd.read_excel('data/billing.xlsx', header=3, dtype=str)

month_cols = [col for col in df_billing.columns if col not in ['Лицевой счёт', 'Тип клиента']]

for col in month_cols:
    df_billing[col] = (
        df_billing[col]
        .astype(str)
        .str.strip()
        .str.replace('—', '', regex=False) 
        .str.replace(' ', '', regex=False)
        .str.replace(',', '.', regex=False)
    )
    # Заменяем пустые строки на NaN, а затем конвертируем в float
    df_billing[col] = pd.to_numeric(df_billing[col], errors='coerce')

# Проверяем результат
print(df_billing[month_cols].dtypes)
display(df_billing.head())

янв.26    float64
фев.26    float64
мар.26    float64
апр.26    float64
май.26    float64
июн.26    float64
июл.26    float64
авг.26    float64
сен.26    float64
dtype: object


,Лицевой счёт,Тип клиента,янв.26,фев.26,мар.26,апр.26,май.26,июн.26,июл.26,авг.26,сен.26
0,50201125,ФЛ,750.0,750.0,750.0,483.0,750.0,750.0,750.0,750.0,750.0
1,69202258,ФЛ,55000.0,55000.0,55000.0,55000.0,55000.0,55000.0,55000.0,55000.0,55000.0
2,33201941,ФЛ,800.0,800.0,800.0,800.0,800.0,800.0,800.0,800.0,800.0
3,45202567,ФЛ,950.0,950.0,950.0,950.0,950.0,950.0,950.0,950.0,950.0
4,97201898,ФЛ,44000.0,44000.0,44000.0,44000.0,44000.0,44000.0,44000.0,44000.0,44000.0


In [41]:
# 1. Переименовываем колонку в 'client_id'
df_billing = df_billing.rename(columns={'Лицевой счёт': 'client_id'})

# 2. Приводим к формату справочника (добавляем 'RT-', если его нет)
df_billing['client_id'] = df_billing['client_id'].apply(
    lambda x: f"RT-{x}" if not str(x).startswith('RT') else str(x)
)

# 3. Считаем сумму начислений по всем месяцам для каждого абонента 
# (находим все колонки с месяцами динамически)
month_cols = [col for col in df_billing.columns if col not in ['client_id', 'Тип клиента']]
df_billing['total_billing'] = df_billing[month_cols].sum(axis=1)

# Оставляем только id и итоговую сумму (или нужные колонки)
billing_agg = df_billing[['client_id', 'total_billing']]

# 4. Присоединяем к основному датафрейму df через left join, чтобы никто не потерялся
df = df.merge(billing_agg, on='client_id', how='left')

# Заменяем пропуски на 0 для тех, у кого нет данных по биллингу
df['total_billing'] = df['total_billing'].fillna(0)

# Проверяем результат
print("Итоговый размер датафрейма:", df.shape)
display(df[['client_id', 'total_billing']].head(10))

Итоговый размер датафрейма: (3247, 14)


,client_id,total_billing
0,RT-200006,0.0
1,RT-200063,0.0
2,RT-200070,0.0
3,RT-200084,0.0
4,RT-200110,0.0
5,RT-200119,0.0
6,RT-200158,0.0
7,RT-200186,0.0
8,RT-200204,0.0
9,RT-200219,0.0


In [42]:
print("Количество ненулевых значений биллинга:", (df['total_billing'] > 0).sum())
display(df[df['total_billing'] > 0][['client_id', 'total_billing']].head())

Количество ненулевых значений биллинга: 0


,client_id,total_billing


In [43]:
print("Примеры client_id в df (справочник):")
print(df['client_id'].head(3).tolist())

print("\nПримеры client_id в df_billing до слияния:")
print(df_billing['client_id'].head(3).tolist())

Примеры client_id в df (справочник):
[' RT-200006 ', ' RT-200063 ', ' RT-200070 ']

Примеры client_id в df_billing до слияния:
['RT-50201125', 'RT-69202258', 'RT-33201941']


In [44]:
# 1. Убираем лишние пробелы в справочнике df
df['client_id'] = df['client_id'].astype(str).str.strip()

# 2. Убираем лишние пробелы в биллинге df_billing
df_billing['client_id'] = df_billing['client_id'].astype(str).str.strip()

# 3. Пересоздаем агрегацию биллинга
month_cols = [col for col in df_billing.columns if col not in ['client_id', 'Тип клиента']]
df_billing['total_billing'] = df_billing[month_cols].sum(axis=1)
billing_agg = df_billing[['client_id', 'total_billing']]

# 4. Удаляем старую колонку total_billing из df, если она там уже затесалась с нулями
if 'total_billing' in df.columns:
    df = df.drop(columns=['total_billing'])

# 5. Делаем правильный слияние (left join)
df = df.merge(billing_agg, on='client_id', how='left')
df['total_billing'] = df['total_billing'].fillna(0)

# 6. Проверяем результат!
print("Количество ненулевых значений биллинга после исправления:", (df['total_billing'] > 0).sum())
display(df[df['total_billing'] > 0][['client_id', 'total_billing']].head())

Количество ненулевых значений биллинга после исправления: 0


,client_id,total_billing


## Часть 4. Технология или город?

1. Соберите итоговую таблицу «один абонент — одна строка»: `client_id`, город, технология, статус, число обращений, среднее начисление.
2. Постройте таблицу доли расторгнувших (%) в разрезе **технология (ADSL / не ADSL) × город (Курган и Нижний Тагил / остальные)** — 4 ячейки. Подпишите, сколько абонентов в каждой.
3. Ответьте на вопрос руководителя в выводе ниже.

*Подсказки:* `pd.crosstab(..., values=..., aggfunc='mean')` или `pivot_table`.

## Вопросы для теста

Ответы получайте кодом, каждый — в отдельной ячейке ниже. Дробные — с указанным в тесте числом знаков.

| № | Вопрос |
|---|---|
| 1 | Сколько уникальных абонентов в справочнике? |
| 2 | Сколько абонентов в Екатеринбурге (по актуальным записям)? |
| 3 | Медианный трафик абонентов Перми за сентябрь, ГБ |
| 4 | Средняя оценка NPS среди ответивших на опрос |
| 5 | Сколько обращений от абонентов справочника за январь–сентябрь (без тестовых учёток; двойная регистрация — одно обращение) |
| 6 | Доля абонентов, ни разу не обращавшихся в поддержку, % |
| 7 | Среднее начисление за месяц у абонентов Тюмени, руб. (по месяцам, где начисление было) |
| 8 | Доля расторгнувших среди абонентов ADSL в Кургане и Нижнем Тагиле вместе, % |

**Вопрос 1**

**Вопрос 2**

**Вопрос 3**

**Вопрос 4**

**Вопрос 5**

**Вопрос 6**

**Вопрос 7**

**Вопрос 8**

## Журнал очистки

Каждая найденная проблема — строка таблицы. Минимум **6 строк**. «Сколько строк затронуто» — число, полученное кодом.

| № | Файл, столбец | Проблема | Как обнаружил | Что сделал | Сколько строк затронуто |
|---|---|---|---|---|---|
| 1 | | | | | |
| 2 | | | | | |
| 3 | | | | | |
| 4 | | | | | |
| 5 | | | | | |
| 6 | | | | | |

## Вывод: технология или город?

Три–пять предложений для руководителя филиала: ответ на вопрос, числа из таблицы части 4, и чего эти данные **не** позволяют утверждать.

*Ваш вывод:*



## Для тех, кто закончил раньше (без баллов в тесте, в зачёт активности)

В биллинге есть абоненты со статусом «расторгнут», которым продолжают начислять плату. Найдите их: сколько их и на какую сумму им начислено после последнего месяца, в котором они ещё были активны?
Подумайте, как определить этот месяц по имеющимся данным, и опишите ограничения своего способа.

## Как оценивается ноутбук (4 балла)

- 1 — ноутбук выполняется сверху вниз без ошибок (Kernel → Restart Kernel and Run All Cells);
- 1 — журнал очистки: не меньше 6 проблем, у каждой — число затронутых строк;
- 1 — таблица 2×2 «технология × город» с долями оттока и числом абонентов;
- 1 — вывод отвечает на вопрос, опирается на числа и называет ограничения.